# PathSnap - Random Forest on Google Colab

Run steps 1 and 2 first. They need no project data and confirm the code works.

Before step 3, put these files in `MyDrive/pathsnap_data`:

- `candidate_feature_table.csv`
- `candidate_labels.csv`
- `candidate_generation_summary.csv`


In [ ]:
# Step 1: get the code and install dependencies
%cd /content
!rm -rf ML_Hackathon
!git clone --depth 1 --branch feature/random-forest https://github.com/ML-Hackathon-PES/ML_Hackathon.git
%cd /content/ML_Hackathon
!pip install -q -r models/random_forest/requirements.txt

In [ ]:
# Step 2: test on synthetic data. This should end with ALL CHECKS PASSED
!python models/random_forest/test_train_rf.py

In [ ]:
# Step 3: connect Google Drive and check the input files
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/pathsnap_data')
for name in ['candidate_feature_table.csv', 'candidate_labels.csv', 'candidate_generation_summary.csv']:
    path = DATA / name
    print(name, f'{path.stat().st_size / 1e6:.0f} MB' if path.exists() else 'MISSING')

In [ ]:
# Step 4: train. Run the baseline first, then set TUNE = True and run this cell again.
# Tuning trains 30 models, so it takes much longer.
TUNE = False

run_name = 'tuned' if TUNE else 'baseline'
out = DATA / 'random_forest_results' / run_name
tune_flag = '' if TUNE else '--no-tune'
!python models/random_forest/train_rf.py --features "{DATA}/candidate_feature_table.csv" --labels "{DATA}/candidate_labels.csv" --candidate-summary "{DATA}/candidate_generation_summary.csv" --out "{out}" {tune_flag}

In [ ]:
# Step 5: show the results, which are also saved in Google Drive
import json

summary = json.loads((out / 'metrics.json').read_text())
print('Hyperparameters:', summary['hyperparameters'])
print('CV mean:', json.dumps(summary['cv_mean'], indent=2))
print('Student test:', json.dumps(summary['student_test_metrics'], indent=2))